# 03 · OkTex Measurement Analytics (Delta)

Analytical queries over `stable_classic_wg38i9_catalog.oneok_okt` powering the dashboard and app — daily system balance, throughput by segment, and meters with the largest scheduled-vs-actual variance. Outputs are live query results.

In [ ]:
from dbx_sql import run_sql, fmt_table, get_token, CATALOG, SCHEMA
FQ = f'{CATALOG}.{SCHEMA}'
tok = get_token()
print('querying', FQ)

querying stable_classic_wg38i9_catalog.oneok_okt


### 1. Daily system balance — receipts vs deliveries (last 10 days)

In [ ]:
rows, cols = run_sql(f'''
  SELECT m.flow_date,
         ROUND(SUM(CASE WHEN d.meter_type='RECEIPT' THEN m.actual_dth END)/1000,1) AS receipts_mdth,
         ROUND(SUM(CASE WHEN d.meter_type IN ('DELIVERY','INTERCONNECT') THEN m.actual_dth END)/1000,1) AS deliv_mdth,
         ROUND((SUM(CASE WHEN d.meter_type='RECEIPT' THEN m.actual_dth END)
               -SUM(CASE WHEN d.meter_type IN ('DELIVERY','INTERCONNECT') THEN m.actual_dth END))/1000,1) AS imbalance_mdth
  FROM {FQ}.fact_daily_measurements m JOIN {FQ}.dim_meters d USING (meter_id)
  WHERE m.flow_date >= (SELECT MAX(flow_date) FROM {FQ}.fact_daily_measurements) - INTERVAL 9 DAYS
  GROUP BY m.flow_date ORDER BY m.flow_date''', tok)
print(fmt_table(rows, cols))

flow_date  | receipts_mdth | deliv_mdth | imbalance_mdth
-----------+---------------+------------+---------------
2026-08-19 | 439.8         | 456.4      | -16.6         
2026-08-20 | 459.3         | 469.0      | -9.7          
2026-08-21 | 459.4         | 450.8      | 8.6           
2026-08-22 | 425.6         | 422.5      | 3.1           
2026-08-23 | 410.7         | 416.2      | -5.5          
2026-08-24 | 448.7         | 441.9      | 6.8           
2026-08-25 | 466.3         | 461.2      | 5.1           
2026-08-26 | 438.6         | 462.9      | -24.2         
2026-08-27 | 464.4         | 472.4      | -8.0          
2026-08-28 | 449.6         | 460.4      | -10.8         


### 2. Throughput by pipeline segment (today)

In [ ]:
rows, cols = run_sql(f'''
  SELECT d.segment, COUNT(*) AS meters, ROUND(SUM(m.actual_dth)/1000,1) AS total_mdth
  FROM {FQ}.fact_daily_measurements m JOIN {FQ}.dim_meters d USING (meter_id)
  WHERE m.flow_date = (SELECT MAX(flow_date) FROM {FQ}.fact_daily_measurements)
  GROUP BY d.segment ORDER BY total_mdth DESC''', tok)
print(fmt_table(rows, cols))

segment       | meters | total_mdth
--------------+--------+-----------
Panhandle     | 6      | 292.3     
West Texas    | 4      | 189.5     
North Central | 5      | 188.4     
Western OK    | 3      | 153.4     
OK Panhandle  | 2      | 86.4      


### 3. Meters with the largest measurement variance (today)

In [ ]:
rows, cols = run_sql(f'''
  SELECT d.meter_id, d.meter_name, d.meter_type, m.scheduled_dth, m.actual_dth, m.variance_pct
  FROM {FQ}.fact_daily_measurements m JOIN {FQ}.dim_meters d USING (meter_id)
  WHERE m.flow_date = (SELECT MAX(flow_date) FROM {FQ}.fact_daily_measurements)
  ORDER BY ABS(m.variance_pct) DESC LIMIT 10''', tok)
print(fmt_table(rows, cols))

meter_id | meter_name             | meter_type   | scheduled_dth | actual_dth | variance_pct
---------+------------------------+--------------+---------------+------------+-------------
OKT-011  | Guymon Interconnect    | INTERCONNECT | 37325         | 35142      | -5.85       
OKT-008  | Carson Receipt         | RECEIPT      | 73202         | 69732      | -4.74       
OKT-014  | Mooreland Delivery     | DELIVERY     | 22045         | 23079      | 4.69        
OKT-001  | Levelland Receipt      | RECEIPT      | 75829         | 72494      | -4.4        
OKT-018  | Medford Interconnect   | INTERCONNECT | 52274         | 54391      | 4.05        
OKT-003  | Plainview Interconnect | INTERCONNECT | 39359         | 37873      | -3.78       
OKT-017  | Enid Terminal          | DELIVERY     | 25578         | 26407      | 3.24        
OKT-019  | Pond Creek Receipt     | RECEIPT      | 49441         | 47884      | -3.15       
OKT-010  | Perryton Receipt       | RECEIPT      | 93385         | 963